In [ ]:
!pip install opendatasets --quiet
!pip install transformers --quiet
import opendatasets as od
od.download("https://www.kaggle.com/datasets/rmisra/news-headlines-dataset-for-sarcasm-detection")

Skipping, found downloaded files in "./news-headlines-dataset-for-sarcasm-detection" (use force=True to force download)


In [ ]:
import torch
import torch.nn as nn
from torch.optim import Adam
from transformers import AutoTokenizer, AutoModel
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
df = pd.read_json('/content/news-headlines-dataset-for-sarcasm-detection/Sarcasm_Headlines_Dataset.json', lines=True)
df.dropna(inplace=True)
df.drop_duplicates(inplace=True)
print(df.shape)
df.head()

(26708, 3)


,article_link,headline,is_sarcastic
0,https://www.huffingtonpost.com/entry/versace-b...,former versace store clerk sues over secret 'b...,0
1,https://www.huffingtonpost.com/entry/roseanne-...,the 'roseanne' revival catches up to our thorn...,0
2,https://local.theonion.com/mom-starting-to-fea...,mom starting to fear son's web series closest ...,1
3,https://politics.theonion.com/boehner-just-wan...,"boehner just wants wife to listen, not come up...",1
4,https://www.huffingtonpost.com/entry/jk-rowlin...,j.k. rowling wishes snape happy birthday in th...,0


In [ ]:
df.drop(columns='article_link', inplace = True)
df

,headline,is_sarcastic
0,former versace store clerk sues over secret 'b...,0
1,the 'roseanne' revival catches up to our thorn...,0
2,mom starting to fear son's web series closest ...,1
3,"boehner just wants wife to listen, not come up...",1
4,j.k. rowling wishes snape happy birthday in th...,0
...,...,...
26704,american politics in moral free-fall,0
26705,america's best 20 hikes,0
26706,reparations and obama,0
26707,israeli ban targeting boycott supporters raise...,0


In [ ]:
X_train , X_temp , y_train , y_temp = train_test_split(np.array(df['headline']), np.array(df['is_sarcastic']), test_size=0.326, random_state=42)
X_val , X_test , y_val , y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")
print(f"X_val shape: {X_val.shape}")
print(f"y_val shape: {y_val.shape}")

X_train shape: (18001,)
X_test shape: (4354,)
y_train shape: (18001,)
y_test shape: (4354,)
X_val shape: (4353,)
y_val shape: (4353,)


In [ ]:
tokenizer = AutoTokenizer.from_pretrained("google-bert/bert-base-uncased")
# bert = AutoModelForMaskedLM.from_pretrained("google-bert/bert-base-uncased")
bert = AutoModel.from_pretrained("google-bert/bert-base-uncased")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
from torch.utils.data import Dataset, DataLoader
class Data(Dataset):
  def __init__(self, X, y):
    self.X = [tokenizer(x, max_length=100, truncation=True, padding="max_length", return_tensors='pt').to(device) for x in X]
    self.y = torch.tensor(y, dtype=torch.float).to(device)

  def __len__(self):
    return len(self.X)

  def __getitem__(self, idx):
    return self.X[idx], self.y[idx]

train_data = Data(X_train, y_train)
val_data = Data(X_val, y_val)
test_data = Data(X_test, y_test)

In [ ]:
BATCH_SIZE = 32
EPOCHS = 10
LR = 1e-4

In [ ]:
train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_data, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_data,  batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
class Zen(nn.Module):
  def __init__(self, bert):
    super(Zen, self).__init__()

    self.bert = bert
    self.drop = nn.Dropout(0.3)
    self.linear1 = nn.Linear(768, 384)
    self.linear2 = nn.Linear(384, 1)
    self.sigmoid = nn.Sigmoid()

  def forward(self, input_ids, attention_mask):
    pooled_output = self.bert(input_ids=input_ids, attention_mask=attention_mask)[0][:,0]
    output = self.linear1(pooled_output)
    output = self.drop(output)
    output = self.linear2(output)
    output = self.sigmoid(output)

    return output

In [ ]:
for params in bert.parameters():
  params.requires_grad = False
model = Zen(bert).to(device)

In [ ]:
loss_fn = nn.BCELoss()
optimizer = Adam(model.parameters(), lr=LR)

In [ ]:
loss_train = []
loss_val = []
acc_train = []
acc_val = []

for epoch in range(EPOCHS):
  total_loss_train = 0
  total_acc_train = 0
  total_loss_val = 0
  total_acc_val = 0

  for idx, data in enumerate(train_loader):
    inputs, labels = data
    inputs.to(device)
    labels.to(device)

    preds = model(inputs['input_ids'].squeeze(1), inputs['attention_mask'].squeeze(1)).squeeze(1)
    loss = loss_fn(preds, labels)
    total_loss_train += loss.item()

    acc = (preds.round() == labels).sum().item()
    total_acc_train += acc

    loss.backward()
    optimizer.step()
    optimizer.zero_grad()

  with torch.no_grad():
    for idx, data in enumerate(val_loader):
      inputs, labels = data
      inputs.to(device)
      labels.to(device)

      preds = model(inputs['input_ids'].squeeze(1), inputs['attention_mask'].squeeze(1)).squeeze(1)

      loss = loss_fn(preds, labels)
      total_loss_val += loss.item()

      acc = (preds.round() == labels).sum().item()
      total_acc_val += acc
  loss_train.append(round(total_loss_train/1000,4))
  loss_val.append(round(total_loss_val/1000,4))
  acc_train.append(round((total_acc_train/train_data.__len__()) * 100,4))
  acc_val.append(round((total_acc_val/train_data.__len__()) * 100,4))

  print(f"Epoch: {epoch+1}/{EPOCHS}")
  print(f"Train Loss: {loss_train[-1]} | Train Acc: {acc_train[-1]}")
  print(f"Val Loss: {loss_val[-1]} | Val Acc: {acc_val[-1]}")

Epoch: 1/10
Train Loss: 0.2055 | Train Acc: 84.5064
Val Loss: 0.0505 | Val Acc: 20.3989
Epoch: 2/10
Train Loss: 0.1925 | Train Acc: 85.0897
Val Loss: 0.0481 | Val Acc: 20.5433
Epoch: 3/10
Train Loss: 0.1855 | Train Acc: 85.8063
Val Loss: 0.0497 | Val Acc: 20.3655
Epoch: 4/10
Train Loss: 0.1814 | Train Acc: 85.9397
Val Loss: 0.046 | Val Acc: 20.6766
Epoch: 5/10
Train Loss: 0.1777 | Train Acc: 86.4119
Val Loss: 0.0458 | Val Acc: 20.7877
Epoch: 6/10
Train Loss: 0.1759 | Train Acc: 86.4841
Val Loss: 0.0467 | Val Acc: 20.7488
Epoch: 7/10
Train Loss: 0.1747 | Train Acc: 86.673
Val Loss: 0.0478 | Val Acc: 20.66
Epoch: 8/10
Train Loss: 0.1729 | Train Acc: 86.7618
Val Loss: 0.0455 | Val Acc: 20.8266
Epoch: 9/10
Train Loss: 0.1718 | Train Acc: 86.6619
Val Loss: 0.0448 | Val Acc: 20.81
Epoch: 10/10
Train Loss: 0.1715 | Train Acc: 86.9007
Val Loss: 0.0453 | Val Acc: 20.8822


In [ ]:
with torch.inference_mode():
  total_acc_test = 0
  total_loss_test = 0
  for idx, data in enumerate(test_loader):
    inputs, labels = data
    inputs.to(device)
    labels.to(device)

    preds = model(inputs['input_ids'].squeeze(1), inputs['attention_mask'].squeeze(1)).squeeze(1)
    loss = loss_fn(preds, labels)
    total_loss_test += loss.item()

    acc = (preds.round() == labels).sum().item()
    total_acc_test += acc

    print(f"Test Loss: {round(total_loss_test/test_data.__len__(),4)} | Test Acc: {round((total_acc_test/test_data.__len__()) * 100,4)}")


In [ ]:
fig, axs = plt.subplot(nrows=1,ncols=2,figsize=(15,5))

axs[0].plot(loss_train, label='Train Loss')
axs[0].plot(loss_val, label='Val Loss')
axs[0].set_title('Training and Validation loss')
axs[0].set_xlabel('Epochs')
axs[0].set_ylabel('Loss')
axs[0].legend()

axs[1].plot(acc_train, label='Train Acc')
axs[1].plot(acc_val, label='Val Acc')
axs[1].set_title('Training and Validation accuracy')
axs[1].set_xlabel('Epochs')
axs[1].set_ylabel('Accuracy')
axs[1].legend()

plt.tight_layout()
plt.show()